### **Evaluación sistemática de modelos visión-lenguaje**

#### **VLMEvalKit, torchmetrics, Winoground y análisis de error**

Este cuaderno propone una evaluación sistemática de modelos visión-lenguaje. El objetivo no es demostrar casos exitosos, sino construir evidencia verificable sobre desempeño, reproducibilidad, composicionalidad visiolingüística, errores y limitaciones.

La lógica del cuaderno viene de la Semana 7, donde se estudian modelos fundacionales y arquitecturas multimodales. En Semana 8 se pregunta cómo evaluar si esas capacidades son reales, medibles y reproducibles.


### **Pregunta experimental**

#### **Formulación**

**Pregunta principal:**

¿Un modelo visión-lenguaje que obtiene buena similitud imagen-texto realmente distingue relaciones composicionales entre objetos, acciones y descripciones?

**Preguntas secundarias:**

1. ¿El modelo empareja correctamente imágenes y captions cuando las mismas palabras aparecen en distinto orden?
2. ¿Los errores se deben más a percepción visual, composición lingüística, OCR, relaciones espaciales, conocimiento externo o evaluación?
3. ¿La métrica automática coincide con el análisis cualitativo humano?
4. ¿Qué limitaciones impiden afirmar confiabilidad general?.


### **Referencias de trabajo**

#### **Lecturas recomendadas**

1. Winoground: Probing Vision and Language Models for Visio-Linguistic Compositionality.
2. Why is Winoground Hard? Investigating Failures in Visio-Linguistic Compositionality.
3. VLMEvalKit: An Open-Source Toolkit for Evaluating Large Multi-Modality Models.
4. LMMs-Eval: Reality Check on the Evaluation of Large Multimodal Models.
5. MME, MMBench, MM-Vet y POPE como contexto metodológico para evaluación multimodal.

Estas lecturas se usan como soporte metodológico. El cuaderno no reemplaza la lectura de los papers.


### **Diseño experimental**

#### **Contrato de la tarea**

La tarea principal es emparejamiento imagen-texto. Para cada caso, existen dos imágenes y dos captions. El modelo debe asignar mayor similitud a los pares correctos.

En un caso tipo Winoground, los dos captions pueden contener palabras muy parecidas o idénticas. La dificultad está en capturar composición, roles, relaciones espaciales, acciones o dependencias finas entre lenguaje e imagen.

#### **Hipótesis inicial**

Un modelo contrastivo puede capturar correspondencias globales imagen-texto, pero fallar cuando la respuesta exige composición fina entre objetos, acciones y relaciones.


### **Configuración reproducible**

#### **Qué se debe registrar**

1. Nombre del modelo.
2. Versión del modelo o checkpoint.
3. Dataset o subconjunto usado.
4. Número de ejemplos.
5. Semilla.
6. Librerías y versiones.
7. Hardware.
8. Parámetros de inferencia.
9. Predicciones crudas.
10. Métricas agregadas.
11. Matriz de errores.


In [ ]:
# Primeras configuraciones
import json
import math
import os
import platform
import random
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

RESULTS_DIR = Path("resultados_semana8")
RESULTS_DIR.mkdir(exist_ok=True)

SEED = 123
random.seed(SEED)
np.random.seed(SEED)

print("Entorno preparado")
print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("Directorio de resultados:", RESULTS_DIR.resolve())


In [ ]:
# Dependencias opcionales.
# El cuaderno puede ejecutarse en modo mínimo sin descargar modelos grandes.

def check_optional_dependencies():
    dependencies = {}

    for module_name in ["torch", "torchmetrics", "transformers", "datasets", "PIL"]:
        try:
            __import__(module_name)
            dependencies[module_name] = "disponible"
        except Exception:
            dependencies[module_name] = "no disponible"

    return dependencies


dependencies = check_optional_dependencies()
pd.DataFrame(
    [{"librería": key, "estado": value} for key, value in dependencies.items()]
)


### **Datos mínimos para pruebas controladas**

#### **Por qué usar un subconjunto pequeño**

Antes de usar un benchmark completo, conviene validar el pipeline con pocos casos controlados. Esto permite comprobar que:

1. Las imágenes se cargan correctamente.
2. Los captions se procesan de forma consistente.
3. La matriz de similitud tiene la orientación correcta.
4. Los resultados crudos se guardan.
5. Los errores se pueden auditar manualmente.

Luego se puede escalar a Winoground real o a VLMEvalKit.


In [ ]:
@dataclass
class PairExample:
    example_id: str
    image_a: str
    image_b: str
    caption_a: str
    caption_b: str
    phenomenon: str
    expected: dict[str, int]


toy_examples = [
    PairExample(
        example_id="toy_001",
        image_a="imagen_a: perro siguiendo a una persona",
        image_b="imagen_b: persona siguiendo a un perro",
        caption_a="un perro sigue a una persona",
        caption_b="una persona sigue a un perro",
        phenomenon="inversión de roles",
        expected={"caption_a": 0, "caption_b": 1},
    ),
    PairExample(
        example_id="toy_002",
        image_a="imagen_a: taza sobre un libro",
        image_b="imagen_b: libro sobre una taza",
        caption_a="una taza está sobre un libro",
        caption_b="un libro está sobre una taza",
        phenomenon="relación espacial",
        expected={"caption_a": 0, "caption_b": 1},
    ),
    PairExample(
        example_id="toy_003",
        image_a="imagen_a: círculo rojo junto a cuadrado azul",
        image_b="imagen_b: cuadrado rojo junto a círculo azul",
        caption_a="un círculo rojo está junto a un cuadrado azul",
        caption_b="un cuadrado rojo está junto a un círculo azul",
        phenomenon="atributos y composición",
        expected={"caption_a": 0, "caption_b": 1},
    ),
]

toy_df = pd.DataFrame([asdict(example) for example in toy_examples])
toy_df


### **Modelo base para ejecución local**

#### **Modo real y modo simulado**

El cuaderno define dos caminos:

1. Modo real: usar un modelo visión-lenguaje compatible con Transformers, por ejemplo CLIP.
2. Modo simulado: usar embeddings deterministas de texto para validar el pipeline cuando no hay GPU, internet o dependencias instaladas.

El modo simulado no evalúa capacidades visuales reales. Solo sirve para probar la estructura del experimento.


In [ ]:
def normalize_matrix(matrix):
    matrix = np.asarray(matrix, dtype=float)
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.maximum(norms, 1e-12)
    return matrix / norms


def deterministic_text_embedding(text, dim=64):
    # Embedding determinista simple para validar el pipeline.
    # No representa comprensión multimodal real.
    rng = np.random.default_rng(abs(hash(text)) % (2**32))
    vector = rng.normal(size=dim)
    return vector / max(np.linalg.norm(vector), 1e-12)


def compute_cosine_matrix(image_embeddings, text_embeddings):
    image_embeddings = normalize_matrix(image_embeddings)
    text_embeddings = normalize_matrix(text_embeddings)
    return image_embeddings @ text_embeddings.T


def build_simulated_embeddings(examples, dim=64):
    image_vectors = []
    text_vectors = []
    metadata = []

    for example in examples:
        image_vectors.append(deterministic_text_embedding(example.image_a, dim=dim))
        image_vectors.append(deterministic_text_embedding(example.image_b, dim=dim))
        text_vectors.append(deterministic_text_embedding(example.caption_a, dim=dim))
        text_vectors.append(deterministic_text_embedding(example.caption_b, dim=dim))
        metadata.append(example.example_id)

    return np.array(image_vectors), np.array(text_vectors), metadata


image_embeddings, text_embeddings, metadata = build_simulated_embeddings(toy_examples)
similarity_matrix = compute_cosine_matrix(image_embeddings, text_embeddings)

pd.DataFrame(
    similarity_matrix,
    index=["img_0", "img_1", "img_2", "img_3", "img_4", "img_5"],
    columns=["txt_0", "txt_1", "txt_2", "txt_3", "txt_4", "txt_5"],
)


### **Carga opcional de CLIP**

#### **Uso con imágenes reales**

La siguiente celda está diseñada para ejecutarse cuando se dispone de internet o de un modelo ya descargado. No es obligatoria para entender el protocolo.

Para una evaluación real, se debe reemplazar el modo simulado por imágenes verdaderas y captions reales.


In [ ]:
def load_clip_model(model_name="openai/clip-vit-base-patch32"):
    try:
        import torch
        from transformers import CLIPModel, CLIPProcessor

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = CLIPModel.from_pretrained(model_name).to(device)
        processor = CLIPProcessor.from_pretrained(model_name)

        print("Modelo cargado:", model_name)
        print("Dispositivo:", device)
        return model, processor, device

    except Exception as exc:
        print("No se pudo cargar CLIP.")
        print("Causa:", str(exc))
        return None, None, "cpu"


# Descomenta para ejecutar con dependencias y acceso al modelo.
# clip_model, clip_processor, clip_device = load_clip_model()


In [ ]:
def encode_clip_images(images, model, processor, device):
    import torch

    inputs = processor(images=images, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        features = model.get_image_features(**inputs)

    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()


def encode_clip_texts(texts, model, processor, device):
    import torch

    inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)

    with torch.no_grad():
        features = model.get_text_features(**inputs)

    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy()


### **Métricas de ranking**

#### **Recuperación imagen-texto**

Para una matriz de similitud, cada fila representa una imagen y cada columna representa un texto. Si el texto correcto para una imagen queda en la primera posición, el ranking es correcto en Recall@1.

Estas métricas son útiles para retrieval, pero no explican por sí solas la naturaleza del error.


In [ ]:
def rank_texts_for_images(similarity):
    rankings = []
    for row in similarity:
        ranking = list(np.argsort(-row))
        rankings.append(ranking)
    return rankings


def compute_recall_at_k(similarity, targets, k=1):
    rankings = rank_texts_for_images(similarity)
    hits = []

    for row_index, target_index in enumerate(targets):
        top_k = rankings[row_index][:k]
        hits.append(int(target_index in top_k))

    return float(np.mean(hits))


# En este ejemplo simulado, asumimos pares diagonales.
targets = list(range(similarity_matrix.shape[0]))

recall_1 = compute_recall_at_k(similarity_matrix, targets, k=1)
recall_2 = compute_recall_at_k(similarity_matrix, targets, k=2)

summary_metrics = {
    "recall_at_1": recall_1,
    "recall_at_2": recall_2,
    "num_pairs": similarity_matrix.shape[0],
}

summary_metrics


### **Uso opcional de torchmetrics**

#### **Qué aporta**

torchmetrics permite estandarizar métricas, reducir errores de implementación y hacer comparaciones más limpias. Sin embargo, la métrica debe corresponder a la tarea.

En tareas de ranking o composición visiolingüística, una accuracy global puede ocultar errores importantes.


In [ ]:
def compute_accuracy_with_torchmetrics(predictions, targets):
    try:
        import torch
        from torchmetrics.classification import MulticlassAccuracy

        predictions_tensor = torch.tensor(predictions)
        targets_tensor = torch.tensor(targets)

        num_classes = int(max(max(predictions), max(targets))) + 1
        metric = MulticlassAccuracy(num_classes=num_classes)
        value = metric(predictions_tensor, targets_tensor)

        return float(value)

    except Exception as exc:
        print("torchmetrics no está disponible o no pudo ejecutarse.")
        print("Causa:", str(exc))
        return None


predicted_top_1 = [int(np.argmax(row)) for row in similarity_matrix]
accuracy_value = compute_accuracy_with_torchmetrics(predicted_top_1, targets)

{
    "predicciones_top_1": predicted_top_1,
    "targets": targets,
    "accuracy_torchmetrics": accuracy_value,
}


### **Evaluación tipo Winoground**

#### **Definición operativa**

Cada ejemplo contiene dos imágenes y dos captions.

Se construye una matriz de similitud de tamaño dos por dos:

1. Fila cero: imagen A.
2. Fila uno: imagen B.
3. Columna cero: caption A.
4. Columna uno: caption B.

La evaluación calcula tres señales:

1. Text score: cada caption debe preferir su imagen correcta.
2. Image score: cada imagen debe preferir su caption correcto.
3. Group score: ambas condiciones deben cumplirse a la vez.

Este criterio es estricto y revela fallas de composición.


In [ ]:
def compute_winoground_scores(similarity_2x2):
    matrix = np.asarray(similarity_2x2, dtype=float)

    image_score = int(matrix[0, 0] > matrix[0, 1] and matrix[1, 1] > matrix[1, 0])
    text_score = int(matrix[0, 0] > matrix[1, 0] and matrix[1, 1] > matrix[0, 1])
    group_score = int(image_score == 1 and text_score == 1)

    return {
        "image_score": image_score,
        "text_score": text_score,
        "group_score": group_score,
        "s00": float(matrix[0, 0]),
        "s01": float(matrix[0, 1]),
        "s10": float(matrix[1, 0]),
        "s11": float(matrix[1, 1]),
    }


def evaluate_toy_winoground(examples):
    rows = []

    for example in examples:
        image_vectors = np.array([
            deterministic_text_embedding(example.image_a),
            deterministic_text_embedding(example.image_b),
        ])
        text_vectors = np.array([
            deterministic_text_embedding(example.caption_a),
            deterministic_text_embedding(example.caption_b),
        ])
        similarity = compute_cosine_matrix(image_vectors, text_vectors)
        scores = compute_winoground_scores(similarity)

        row = {
            "example_id": example.example_id,
            "phenomenon": example.phenomenon,
            **scores,
        }
        rows.append(row)

    return pd.DataFrame(rows)


winoground_toy_results = evaluate_toy_winoground(toy_examples)
winoground_toy_results


### **Interpretación de resultados Winoground**

#### **Qué mirar más allá del promedio**

Un promedio bajo no basta. Se debe inspeccionar qué tipo de fenómeno produce el fallo.

Categorías sugeridas:

1. Inversión de roles.
2. Relación espacial.
3. Atributos asociados al objeto incorrecto.
4. Acción ambigua.
5. Objeto pequeño o difícil de percibir.
6. Dependencia de conocimiento común.
7. Composición lingüística.
8. Fusión imagen-texto insuficiente.
9. Error de evaluación.


In [ ]:
ERROR_CATEGORIES = [
    "error_perceptual",
    "error_espacial",
    "error_composicional",
    "error_ocr",
    "error_conocimiento",
    "error_grounding",
    "error_prompt",
    "error_evaluacion",
]

SEVERITY_LEVELS = ["baja", "media", "alta"]


def make_error_record(
    example_id,
    category,
    affected_modality,
    severity,
    evidence,
    comment,
    mitigation,
):
    if category not in ERROR_CATEGORIES:
        raise ValueError("Categoría de error no reconocida")
    if severity not in SEVERITY_LEVELS:
        raise ValueError("Severidad no reconocida")

    return {
        "example_id": example_id,
        "category": category,
        "affected_modality": affected_modality,
        "severity": severity,
        "evidence": evidence,
        "comment": comment,
        "mitigation": mitigation,
    }


error_records = [
    make_error_record(
        example_id="toy_001",
        category="error_composicional",
        affected_modality="imagen_texto",
        severity="alta",
        evidence="El modelo no distingue quién sigue a quién.",
        comment="La bolsa de palabras es similar, pero la relación sujeto objeto cambia.",
        mitigation="Agregar evaluación composicional y prompts de verificación.",
    ),
    make_error_record(
        example_id="toy_002",
        category="error_espacial",
        affected_modality="imagen",
        severity="media",
        evidence="El modelo confunde objeto superior e inferior.",
        comment="La relación sobre debajo es central para la respuesta.",
        mitigation="Evaluar con ejemplos espaciales balanceados.",
    ),
]

error_df = pd.DataFrame(error_records)
error_df


### **Guardado de salidas**

#### **Reproducibilidad mínima**

Toda evaluación debe guardar predicciones crudas, métricas agregadas y matriz de errores. Sin salidas sin procesar no es posible auditar errores ni repetir el análisis.


In [ ]:
def save_json(data, path):
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)


def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as file:
        for row in rows:
            file.write(json.dumps(row, ensure_ascii=False) + "\n")


config = {
    "experiment": {
        "name": "semana8_evaluacion_sistematica_vlm",
        "seed": SEED,
        "task": "emparejamiento_imagen_texto",
        "dataset": "subconjunto_controlado_tipo_winoground",
        "num_examples": len(toy_examples),
        "mode": "simulado_para_validacion_de_pipeline",
    },
    "runtime": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
    },
    "outputs": {
        "summary": str(RESULTS_DIR / "summary.json"),
        "predictions": str(RESULTS_DIR / "predictions.jsonl"),
        "errors": str(RESULTS_DIR / "error_matrix.csv"),
    },
}

predictions_rows = winoground_toy_results.to_dict(orient="records")

save_json(config, RESULTS_DIR / "config.json")
save_json(summary_metrics, RESULTS_DIR / "summary.json")
save_jsonl(predictions_rows, RESULTS_DIR / "predictions.jsonl")
error_df.to_csv(RESULTS_DIR / "error_matrix.csv", index=False)

print("Archivos guardados:")
for path in sorted(RESULTS_DIR.iterdir()):
    print(path)


### **Carga opcional de Winoground real**

#### **Uso con datasets**

La siguiente celda es opcional. Requiere la librería datasets y acceso al dataset correspondiente. La estructura exacta puede cambiar según la versión del dataset, por lo que siempre se debe inspeccionar una muestra antes de evaluar.

El objetivo de esta sección es mostrar cómo pasar del subconjunto controlado al benchmark real.


In [ ]:
def load_winoground_dataset(split="test"):
    try:
        from datasets import load_dataset

        dataset = load_dataset("facebook/winoground", split=split)
        print("Dataset cargado:", dataset)
        print("Columnas:", dataset.column_names)
        return dataset

    except Exception as exc:
        print("No se pudo cargar Winoground.")
        print("Causa:", str(exc))
        return None


# Descomenta si se cuenta con acceso al dataset.
# winoground_dataset = load_winoground_dataset()
# winoground_dataset[0]


### **Evaluación real de Winoground con CLIP**

#### **Plantilla de implementación**

Esta sección muestra la estructura necesaria para evaluar ejemplos reales. Debe adaptarse a los nombres de columnas que entregue el dataset.

La evaluación real requiere imágenes PIL, captions y un modelo cargado.


In [ ]:
def evaluate_winoground_example_with_clip(example, model, processor, device):
    # Esta función es una plantilla.
    # Ajustar nombres de campos según la estructura real del dataset.

    image_0 = example["image_0"]
    image_1 = example["image_1"]
    caption_0 = example["caption_0"]
    caption_1 = example["caption_1"]

    image_embeddings = encode_clip_images([image_0, image_1], model, processor, device)
    text_embeddings = encode_clip_texts([caption_0, caption_1], model, processor, device)

    similarity = compute_cosine_matrix(image_embeddings, text_embeddings)
    scores = compute_winoground_scores(similarity)

    return {
        "id": example.get("id", "sin_id"),
        "caption_0": caption_0,
        "caption_1": caption_1,
        **scores,
    }


def evaluate_winoground_dataset_with_clip(dataset, model, processor, device, max_examples=20):
    rows = []

    for index, example in enumerate(dataset):
        if index >= max_examples:
            break

        try:
            row = evaluate_winoground_example_with_clip(example, model, processor, device)
            rows.append(row)
        except Exception as exc:
            rows.append({
                "id": example.get("id", f"ejemplo_{index}"),
                "error": str(exc),
            })

    return pd.DataFrame(rows)


### **Discusión: Winoground Findings**

#### **Por qué puede fallar un modelo fuerte**

Los resultados de Winoground deben interpretarse con análisis cualitativo. Un fallo puede deberse a múltiples causas:

1. El modelo reconoce objetos, pero no relaciones.
2. El modelo usa palabras clave y no composición.
3. El objeto relevante es pequeño o ambiguo.
4. La imagen requiere conocimiento común.
5. El caption contiene una estructura sintáctica difícil.
6. La similitud global no penaliza inversión de roles.
7. La evaluación binaria no captura aciertos parciales.

Por eso el análisis de error es obligatorio.


### **Extensión opcional: evaluación generativa con prompts tipo Winoground**

#### **DALL-E 2 on Winoground como discusión**

Si se dispone de un modelo texto-imagen, se pueden usar captions tipo Winoground como prompts y evaluar si la imagen generada respeta la composición esperada.

No se debe mezclar esta extensión con el score original de Winoground. La evaluación generativa responde otra pregunta:

¿El modelo texto-imagen genera una escena que respeta objetos, atributos, roles y relaciones del prompt?

Criterios sugeridos:

1. Presencia de objetos correctos.
2. Atributos asignados al objeto correcto.
3. Relación espacial correcta.
4. Rol o acción correcta.
5. Ausencia de objetos inventados relevantes.
6. Coherencia visual suficiente.


In [ ]:
def evaluate_generated_image_record(
    prompt,
    generated_image_id,
    objects_correct,
    attributes_correct,
    spatial_relation_correct,
    role_correct,
    hallucinated_objects,
    human_comment,
):
    # Registro manual para evaluar imágenes generadas con prompts composicionales.
    return {
        "prompt": prompt,
        "generated_image_id": generated_image_id,
        "objects_correct": bool(objects_correct),
        "attributes_correct": bool(attributes_correct),
        "spatial_relation_correct": bool(spatial_relation_correct),
        "role_correct": bool(role_correct),
        "hallucinated_objects": hallucinated_objects,
        "human_comment": human_comment,
    }


generated_eval_example = evaluate_generated_image_record(
    prompt="un perro sigue a una persona",
    generated_image_id="generacion_001",
    objects_correct=True,
    attributes_correct=True,
    spatial_relation_correct=False,
    role_correct=False,
    hallucinated_objects="ninguno",
    human_comment="La imagen contiene perro y persona, pero la acción no está clara.",
)

generated_eval_example


### **VLMEvalKit**

#### **Cuándo usarlo**

VLMEvalKit es útil cuando se necesita evaluar modelos multimodales con una interfaz común y benchmarks estandarizados. Para este cuaderno, se presenta como extensión avanzada porque puede requerir instalación, modelos grandes, GPU, datos externos y configuración específica.

Uso recomendado:

1. Primero validar el protocolo con un subconjunto pequeño.
2. Luego ejecutar benchmarks con VLMEvalKit.
3. Guardar comandos, configuración, versiones y resultados.
4. Comparar modelos bajo el mismo pipeline.
5. Auditar errores representativos, no solo resultados agregados.


In [ ]:
# Plantilla de comandos para documentar una evaluación con VLMEvalKit.
# No se ejecuta desde este cuaderno por defecto.

vlmevalkit_plan = {
    "objetivo": "Evaluar un modelo visión-lenguaje con benchmark estandarizado",
    "pasos": [
        "Crear entorno aislado",
        "Instalar VLMEvalKit según documentación oficial",
        "Seleccionar modelo y dataset",
        "Ejecutar evaluación con configuración fija",
        "Guardar logs, resultados y salidas crudas",
        "Auditar errores representativos",
    ],
    "evidencia_minima": [
        "comando ejecutado",
        "commit o versión de la herramienta",
        "modelo y checkpoint",
        "dataset y partición",
        "número de muestras",
        "métricas reportadas",
        "matriz de errores",
    ],
}

vlmevalkit_plan


### **Actividad de cierre**

#### **Ejercicio 1: Pregunta experimental**

Formula una pregunta experimental para evaluar un modelo visión-lenguaje en una de estas tareas:

1. Emparejamiento imagen-texto.
2. Recuperación imagen-texto.
3. Captioning.
4. VQA.
5. Grounding.
6. Evaluación tipo Winoground.

La pregunta debe incluir modalidad de entrada, salida esperada, capacidad evaluada y condición de fallo.

#### **Ejercicio 2: Selección de casos de prueba**

Selecciona al menos diez ejemplos. Cada ejemplo debe quedar asociado a una categoría de fenómeno.

Categorías sugeridas:

1. Relación espacial.
2. Inversión de roles.
3. Atributos asociados a objetos.
4. Conteo.
5. OCR.
6. Objeto pequeño.
7. Ambigüedad visual.
8. Conocimiento externo.
9. Grounding.
10. Alucinación visual.

#### **Ejercicio 3: Evaluación cuantitativa**

Ejecuta el pipeline del cuaderno y reporte al menos una métrica principal y una métrica secundaria.

Debes indicar qué mide cada métrica y qué no mide. Por ejemplo, una similitud imagen-texto puede capturar alineamiento global, pero no demuestra por sí sola razonamiento composicional.

#### **Ejercicio 4: Matriz de errores**

Selecciona al menos cinco errores o casos dudosos y clasifíquelos.

Cada registro debe incluir:

1. Identificador del caso.
2. Predicción del modelo.
3. Respuesta esperada.
4. Tipo de error.
5. Modalidad afectada.
6. Evidencia observada.
7. Severidad.
8. Posible causa.
9. Mitigación propuesta.

#### **Ejercicio 5: Conclusión responsable**

Redacta una conclusión breve que responda directamente la pregunta experimental.

La conclusión debe indicar:

1. Qué se demostró.
2. Qué no se demostró.
3. Bajo qué condiciones se obtuvieron los resultados.
4. Qué errores fueron más importantes.
5. Qué experimento debería realizarse después.


### **Formato de entrega del mini-informe experimental**

#### **Qué se debe entregar**

Debes entregar un mini-informe experimental breve, acompañado por los archivos generados durante el cuaderno. Este informe no es un resumen libre ni una opinión general sobre el modelo. Es un documento técnico que debe demostrar qué se evaluó, cómo se evaluó, qué resultados se obtuvieron, qué errores aparecieron y qué conclusiones están justificadas por la evidencia.

#### **1. Problema y pregunta experimental**

Debe describirse el problema evaluado y formularse una pregunta experimental específica.

Debe incluir:

1. Tarea multimodal evaluada.
2. Modalidades involucradas.
3. Capacidad que se quiere medir.
4. Pregunta experimental.
5. Hipótesis o expectativa inicial.
6. Condición bajo la cual el modelo se consideraría insuficiente.

#### **2. Modelo evaluado**

Debe identificarse con precisión el modelo usado.

Debe incluir:

1. Nombre del modelo.
2. Versión, checkpoint o repositorio.
3. Tipo de arquitectura.
4. Modalidades de entrada y salida.
5. Razón de selección.
6. Limitaciones conocidas.
7. Restricciones de inferencia relevantes.

#### **3. Datos, partición y criterios de selección**

Debe describirse el conjunto de datos o subconjunto usado.

Debe incluir:

1. Fuente de los datos.
2. Número de ejemplos.
3. Partición usada.
4. Criterios de filtrado.
5. Categorías o fenómenos incluidos.
6. Posibles sesgos del subconjunto.
7. Ejemplos representativos.
8. Casos excluidos y razón de exclusión.

#### **4. Métricas y justificación**

Debe explicarse qué métricas se usan y por qué corresponden a la tarea.

Debe incluir:

1. Métrica principal.
2. Métricas secundarias.
3. Definición operativa de cada métrica.
4. Relación entre métrica y pregunta experimental.
5. Qué aspecto no mide cada métrica.
6. Razón para complementar métricas automáticas con análisis cualitativo.

#### **5. Configuración reproducible**

Debe documentarse cómo se ejecutó el experimento.

Debe incluir:

1. Versión de Python.
2. Librerías principales.
3. Modelo y checkpoint.
4. Semilla.
5. Hardware usado.
6. Parámetros de inferencia.
7. Comandos o celdas ejecutadas.
8. Archivos generados.
9. Ruta de predicciones crudas.
10. Ruta de resultados agregados.
11. Ruta de matriz de errores.

#### **6. Resultados cuantitativos**

Debe presentarse una tabla clara con los resultados.

Debe incluir:

1. Número total de ejemplos.
2. Resultados globales.
3. Resultados por categoría o fenómeno.
4. Comparación con baseline si existe.
5. Comentario sobre tamaño muestral.
6. Casos donde la métrica puede ser insuficiente.

#### **7. Análisis de errores**

Debe analizarse un conjunto de errores representativos.

Debe incluir:

1. Identificador del ejemplo.
2. Predicción del modelo.
3. Respuesta esperada.
4. Tipo de error.
5. Modalidad afectada.
6. Evidencia observada.
7. Severidad.
8. Posible causa.
9. Mitigación propuesta.

Categorías sugeridas:

1. Error perceptual.
2. Error espacial.
3. Error composicional.
4. Error de OCR.
5. Error de conocimiento.
6. Error de grounding.
7. Alucinación visual.
8. Error de prompt.
9. Error de evaluación.

#### **8. Discusión de limitaciones**

Debe explicarse qué no permite concluir el experimento.

Debe incluir:

1. Limitaciones del tamaño de muestra.
2. Limitaciones del dataset.
3. Limitaciones del modelo.
4. Limitaciones de las métricas.
5. Dependencia del prompt.
6. Posible contaminación del benchmark.
7. Restricciones de cómputo.
8. Amenazas a la validez experimental.

#### **9. Riesgos de confiabilidad**

Debe discutirse si el sistema puede inducir conclusiones equivocadas.

Debe incluir:

1. Riesgo de alucinación.
2. Riesgo de sobreconfianza.
3. Riesgo de falta de grounding.
4. Riesgo de error sistemático por categoría.
5. Riesgo de uso fuera del alcance.
6. Necesidad de supervisión humana.

#### **10. Conclusión proporcional a la evidencia**

La conclusión debe responder directamente la pregunta experimental.

Debe incluir:

1. Qué se demostró.
2. Qué no se demostró.
3. Bajo qué condiciones se obtuvieron los resultados.
4. Qué errores fueron más relevantes.
5. Qué mejora o experimento siguiente se recomienda.

#### **Archivos esperados**

El trabajo final debe incluir:

1. Mini-informe en Markdown o PDF.
2. Archivo de configuración experimental.
3. Predicciones crudas en JSONL o CSV.
4. Tabla de resultados agregados.
5. Matriz de errores.
6. Discusión de limitaciones.
7. Conclusión responsable.


In [ ]:
def build_report_template():
    # Plantilla estructurada para el mini-informe experimental.
    report = {
        "problema_y_pregunta_experimental": {
            "tarea_multimodal": "",
            "modalidades": [],
            "capacidad_evaluada": "",
            "pregunta_experimental": "",
            "hipotesis_inicial": "",
            "condicion_de_fallo": "",
        },
        "modelo_evaluado": {
            "nombre": "",
            "version_o_checkpoint": "",
            "arquitectura": "",
            "entradas": [],
            "salidas": [],
            "razon_de_seleccion": "",
            "limitaciones_conocidas": [],
        },
        "datos": {
            "fuente": "",
            "particion": "",
            "numero_de_ejemplos": 0,
            "criterios_de_filtrado": [],
            "categorias_o_fenomenos": [],
            "posibles_sesgos": [],
            "casos_excluidos": [],
        },
        "metricas": {
            "principal": "",
            "secundarias": [],
            "definiciones_operativas": {},
            "justificacion": "",
            "aspectos_no_medidos": [],
        },
        "configuracion_reproducible": {
            "python": sys.version.split()[0],
            "semilla": SEED,
            "hardware": "",
            "librerias": dependencies,
            "modelo": "",
            "parametros_inferencia": {},
            "archivos_generados": {
                "configuracion": str(RESULTS_DIR / "config.json"),
                "predicciones": str(RESULTS_DIR / "predictions.jsonl"),
                "resumen": str(RESULTS_DIR / "summary.json"),
                "matriz_errores": str(RESULTS_DIR / "error_matrix.csv"),
            },
        },
        "resultados_cuantitativos": {
            "resultados_globales": {},
            "resultados_por_categoria": {},
            "baseline": "",
            "comentario_sobre_tamano_muestral": "",
        },
        "analisis_de_errores": [],
        "limitaciones": {
            "tamano_muestra": "",
            "dataset": "",
            "modelo": "",
            "metricas": "",
            "prompt": "",
            "computo": "",
            "amenazas_a_la_validez": [],
        },
        "riesgos_de_confiabilidad": {
            "alucinacion": "",
            "sobreconfianza": "",
            "falta_de_grounding": "",
            "error_sistematico": "",
            "uso_fuera_del_alcance": "",
            "supervision_humana": "",
        },
        "conclusion": {
            "que_se_demostro": "",
            "que_no_se_demostro": "",
            "condiciones_del_resultado": "",
            "errores_mas_relevantes": "",
            "siguiente_experimento": "",
        },
    }

    return report


def build_error_audit_template():
    # Tabla base para auditar errores representativos.
    columns = [
        "example_id",
        "prediccion_modelo",
        "respuesta_esperada",
        "tipo_error",
        "modalidad_afectada",
        "evidencia_observada",
        "severidad",
        "posible_causa",
        "mitigacion_propuesta",
    ]

    return pd.DataFrame(columns=columns)


def build_submission_checklist():
    # Checklist de control antes de cerrar el cuaderno.
    rows = [
        {"criterio": "pregunta experimental especifica", "cumple": False, "evidencia": ""},
        {"criterio": "modelo identificado con version o checkpoint", "cumple": False, "evidencia": ""},
        {"criterio": "datos y particion documentados", "cumple": False, "evidencia": ""},
        {"criterio": "metricas justificadas", "cumple": False, "evidencia": ""},
        {"criterio": "configuracion reproducible registrada", "cumple": False, "evidencia": ""},
        {"criterio": "predicciones crudas guardadas", "cumple": False, "evidencia": ""},
        {"criterio": "resultados cuantitativos reportados", "cumple": False, "evidencia": ""},
        {"criterio": "matriz de errores completada", "cumple": False, "evidencia": ""},
        {"criterio": "limitaciones discutidas", "cumple": False, "evidencia": ""},
        {"criterio": "conclusion proporcional a la evidencia", "cumple": False, "evidencia": ""},
    ]

    return pd.DataFrame(rows)


report_template = build_report_template()
error_audit_template = build_error_audit_template()
submission_checklist = build_submission_checklist()

save_json(report_template, RESULTS_DIR / "report_template.json")
error_audit_template.to_csv(RESULTS_DIR / "error_audit_template.csv", index=False)
submission_checklist.to_csv(RESULTS_DIR / "submission_checklist.csv", index=False)

print("Archivos de cierre generados:")
print(RESULTS_DIR / "report_template.json")
print(RESULTS_DIR / "error_audit_template.csv")
print(RESULTS_DIR / "submission_checklist.csv")

submission_checklist


#### **Mensaje central**

La evaluación multimodal no debe reducirse a una métrica agregada. Un modelo puede recuperar captions razonables y aun así fallar en composición, roles, relaciones espaciales, grounding o robustez.

Un cuaderno de evaluación sistemática debe producir tres tipos de evidencia:

1. Evidencia cuantitativa mediante métricas coherentes con la tarea.
2. Evidencia cualitativa mediante análisis de error.
3. Evidencia reproducible mediante configuración, versiones y salidas crudas.

La conclusión académica debe declarar qué se demostró, qué no se demostró y bajo qué condiciones se obtuvieron los resultados.

